# BÀI TẬP 1: TÌM TẬP PHỔ BIẾN TRONG DỮ LIỆU SIÊU THỊ
Mục tiêu: Sử dụng giải thuật Apriori từ thư viện `mlxtend` để tìm các tập sản phẩm thường xuyên được mua cùng nhau (tập phổ biến) từ dữ liệu `Online Retail`.

## 1. Cài đặt và Import thư viện

Cài đặt thư viện mlxtend nếu chưa có (bỏ dấu `#` ở dòng pip install).

In [7]:
# !pip install mlxtend pandas matplotlib
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import matplotlib.pyplot as plt

## 2. Nạp và Tiền xử lý dữ liệu

Dữ liệu thực tế thường chứa nhiễu (giá trị rỗng, giao dịch hủy...). Cần làm sạch trước khi đưa vào thuật toán.

In [8]:
# Đọc dữ liệu từ file CSV
df = pd.read_excel("Online Retail.xlsx")

# Hiển thị thông tin sơ bộ
print("Kích thước dữ liệu gốc:", df.shape)

# --- Làm sạch dữ liệu ---
# 1. Loại bỏ các dòng có 'Description' bị trống
df = df.dropna(axis=0, subset=['Description'])

# 2. Chuyển đổi cột 'InvoiceNo' sang dạng chuỗi
df['InvoiceNo'] = df['InvoiceNo'].astype('str')

# 3. Loại bỏ các giao dịch tín dụng (những hóa đơn bắt đầu bằng 'C' - Cancelled)
df = df[~df['InvoiceNo'].str.contains('C')]

# 4. Chuẩn hóa chuỗi (xóa khoảng trắng thừa ở đầu/cuối mô tả sản phẩm)
df['Description'] = df['Description'].str.strip()

print("Kích thước dữ liệu sau khi làm sạch:", df.shape)

Kích thước dữ liệu gốc: (541909, 8)
Kích thước dữ liệu sau khi làm sạch: (531167, 8)


## 3. Tạo Giỏ hàng
Thuật toán Apriori yêu cầu dữ liệu đầu vào dạng ma trận: Hàng là hóa đơn (Invoice), Cột là sản phẩm (Item), Giá trị là 0 hoặc 1 (có mua hoặc không).

Lưu ý: Để demo nhanh và tránh tràn bộ nhớ với tập dữ liệu lớn này, chúng ta sẽ lọc lấy dữ liệu của một quốc gia cụ thể (ví dụ: "France").

In [9]:
# Lọc dữ liệu của Pháp (France) để làm mẫu
basket = (df[df['Country'] == "France"]
          .groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

# Hàm chuyển đổi dữ liệu số lượng thành dạng nhị phân (0 hoặc 1)
# Nếu số lượng > 0 thì là 1 (có mua), ngược lại là 0
def encode_units(x):
    if x <= 0:
        return False
    if x >= 1:
        return True

# Áp dụng hàm mã hóa
basket_sets = basket.applymap(encode_units)

# Loại bỏ cột 'POSTAGE' (tiền cước phí) nếu có, vì nó không phải là sản phẩm
if 'POSTAGE' in basket_sets.columns:
    basket_sets.drop('POSTAGE', inplace=True, axis=1)

print("\nMa trận giao dịch (Basket Sets):")
print(basket_sets.head())

C:\Users\chinh\AppData\Local\Temp\ipykernel_22520\3105390773.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_sets = basket.applymap(encode_units)



Ma trận giao dịch (Basket Sets):
Description  10 COLOUR SPACEBOY PEN  12 COLOURED PARTY BALLOONS  \
InvoiceNo                                                         
536370                        False                       False   
536852                        False                       False   
536974                        False                       False   
537065                        False                       False   
537463                        False                       False   

Description  12 EGG HOUSE PAINTED WOOD  12 MESSAGE CARDS WITH ENVELOPES  \
InvoiceNo                                                                 
536370                           False                            False   
536852                           False                            False   
536974                           False                            False   
537065                           False                            False   
537463                           False        

## 4. Tìm Tập phổ biến với Apriori
Áp dụng thuật toán Apriori để tìm các nhóm sản phẩm có độ hỗ trợ (support) lớn hơn ngưỡng quy định.

In [10]:
# Thiết lập ngưỡng support tối thiểu (ví dụ: 7% số đơn hàng chứa tập sản phẩm này)
min_support = 0.07

# Tìm tập phổ biến
# use_colnames=True để hiển thị tên sản phẩm thay vì chỉ số cột
frequent_itemsets = apriori(basket_sets, min_support=min_support, use_colnames=True)

# Thêm cột 'length' để biết tập phổ biến chứa bao nhiêu sản phẩm
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

# Sắp xếp kết quả theo độ hỗ trợ giảm dần
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)

print("\nCác tập phổ biến tìm được (Top 10):")
print(frequent_itemsets.head(10))


Các tập phổ biến tìm được (Top 10):
     support                              itemsets  length
22  0.188776                  (RABBIT NIGHT LIGHT)       1
26  0.181122       (RED TOADSTOOL LED NIGHT LIGHT)       1
21  0.170918    (PLASTERS IN TIN WOODLAND ANIMALS)       1
18  0.168367       (PLASTERS IN TIN CIRCUS PARADE)       1
30  0.158163  (ROUND SNACK BOXES SET OF4 WOODLAND)       1
11  0.153061             (LUNCH BAG RED RETROSPOT)       1
14  0.142857    (LUNCH BOX WITH CUTLERY RETROSPOT)       1
19  0.137755            (PLASTERS IN TIN SPACEBOY)       1
33  0.137755         (SET/6 RED SPOTTY PAPER CUPS)       1
24  0.137755            (RED RETROSPOT MINI CASES)       1
